# 2. Импорты и конфиг

In [34]:
# Стандартная библиотека
import gc
import logging
import os
import pickle
import sys
import warnings
from collections import defaultdict
from pathlib import Path
from typing import List, Tuple
import os
import sys
import warnings

# Работа с данными
import numpy as np
import pandas as pd
import polars as pl

# Визуализация
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots

# Статистика
from scipy.sparse import csr_matrix
from scipy.stats import gaussian_kde

# ML
import lightgbm as lgb
from implicit.als import AlternatingLeastSquares
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Файлы и хранилища
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
from dotenv import load_dotenv

# Прочее
import phik
from phik import report, resources
from tqdm.auto import tqdm

In [35]:
# настройки отображения
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
# отключаем научную нотацию для удобаства анализа данных
pd.options.display.float_format = '{:,.0f}'.format

# настройки графиков
%matplotlib inline
%config InlineBackend.figure_format = "retina"

# корень проекта
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PROJECT_ROOT:", os.path.basename(PROJECT_ROOT))
print("src exists:", os.path.isdir(os.path.join(PROJECT_ROOT, "src")))

def to_relative(path, base):
    try:
        return os.path.relpath(path, base)
    except ValueError:
        return path

from src.utils.config import (
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    EVENTS_PATH,
    CATEGORY_TREE_PATH,
    ITEM_PROPERTIES_PATH,
    ARTIFACTS_DIR,
    MODELS_DIR,
    MLFLOW_BASE_DIR,
    MLFLOW_DIR,
    MLFLOW_DB_PATH,
    AIRFLOW_DIR,
    AIRFLOW_DAGS_DIR,
)

# S3
S3_BASE = "s3://s3-student-mle-20250927-31ecef0a74/recsys"
S3_DATA_DIR = f"{S3_BASE}/data"
S3_REC_DIR = f"{S3_BASE}/recommendations"

# проверка путей
paths_to_check = {
    "DATA_DIR": DATA_DIR,
    "RAW_DIR": RAW_DIR,
    "PROCESSED_DIR": PROCESSED_DIR,
    "ARTIFACTS_DIR": ARTIFACTS_DIR,
    "MODELS_DIR": MODELS_DIR,
    "MLFLOW_BASE_DIR": MLFLOW_BASE_DIR,
    "MLFLOW_DIR": MLFLOW_DIR,
    "MLFLOW_DB_PATH": MLFLOW_DB_PATH,
    "AIRFLOW_DIR": AIRFLOW_DIR,
    "AIRFLOW_DAGS_DIR": AIRFLOW_DAGS_DIR,
    "EVENTS_PATH": EVENTS_PATH,
    "CATEGORY_TREE_PATH": CATEGORY_TREE_PATH,
    "ITEM_PROPERTIES_PATH": ITEM_PROPERTIES_PATH,
}

print("\nProject paths:\n")

for name, path in paths_to_check.items():
    rel_path = to_relative(path, PROJECT_ROOT)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"{name:<22} {rel_path:<40} [{status}]")

PROJECT_ROOT: ecommerce-recsys
src exists: True

Project paths:

DATA_DIR               data                                     [OK]
RAW_DIR                data/raw                                 [OK]
PROCESSED_DIR          data/processed                           [OK]
ARTIFACTS_DIR          artifacts                                [OK]
MODELS_DIR             artifacts/models                         [OK]
MLFLOW_BASE_DIR        mlflow                                   [OK]
MLFLOW_DIR             mlflow/mlruns                            [OK]
MLFLOW_DB_PATH         mlflow/mlflow.db                         [OK]
AIRFLOW_DIR            airflow                                  [OK]
AIRFLOW_DAGS_DIR       airflow/dags                             [OK]
EVENTS_PATH            data/raw/events.csv                      [OK]
CATEGORY_TREE_PATH     data/raw/category_tree.csv               [OK]
ITEM_PROPERTIES_PATH   data/raw/item_properties.csv             [MISSING]


# 3. Загрузка данных

In [36]:
import polars as pl

events = pl.read_csv(
    EVENTS_PATH,
    dtypes={
        "visitorid": pl.Int32,
        "event": pl.Categorical,
        "itemid": pl.Int32,
        "transactionid": pl.Float64,
    },
)

category_tree = pl.read_csv(
    f"{RAW_DIR}/category_tree.csv",
    dtypes={
        "categoryid": pl.Int32,
        "parentid": pl.Float64,
    },
)

item_props_part1 = pl.read_csv(
    f"{RAW_DIR}/item_properties_part1.csv",
    dtypes={
        "itemid": pl.Int32,
        "property": pl.Categorical,
        "value": pl.Utf8,
    },
)

item_props_part2 = pl.read_csv(
    f"{RAW_DIR}/item_properties_part2.csv",
    dtypes={
        "itemid": pl.Int32,
        "property": pl.Categorical,
        "value": pl.Utf8,
    },
)

item_props = pl.concat([item_props_part1, item_props_part2])

print("Events shape:", events.shape)
print("Category tree shape:", category_tree.shape)
print("Item properties shape:", item_props.shape)

Events shape: (2756101, 5)
Category tree shape: (1669, 2)
Item properties shape: (20275902, 4)


# 3. Предобработка данных

На этом этапе:
- выполняется приведение типов;
- timestamp переводится в datetime;
- из item_properties извлекаются ключевые признаки товаров;
- формируется таблица item-level признаков для дальнейшего моделирования.

## 3.1. Приведение времени и типов

In [37]:
events = events.with_columns(
    [
        pl.from_epoch("timestamp", time_unit="ms").alias("timestamp_dt"),
        pl.col("visitorid").cast(pl.Int32),
        pl.col("itemid").cast(pl.Int32),
        pl.col("event").cast(pl.Categorical),
    ]
)

item_props = item_props.with_columns(
    [
        pl.from_epoch("timestamp", time_unit="ms").alias("timestamp_dt"),
        pl.col("itemid").cast(pl.Int32),
        pl.col("property").cast(pl.Categorical),
    ]
)

category_tree = category_tree.with_columns(
    [
        pl.col("categoryid").cast(pl.Int32),
        pl.col("parentid").cast(pl.Float32, strict=False),
    ]
)

print(events.dtypes)
print(item_props.dtypes)
print(category_tree.dtypes)

[Int64, Int32, Categorical, Int32, Float64, Datetime(time_unit='us', time_zone=None)]
[Int64, Int32, Categorical, String, Datetime(time_unit='us', time_zone=None)]
[Int32, Float32]


## 3.2. Удаление обрезанного хвоста

In [38]:
CUTOFF_DATE = pl.datetime(2015, 9, 15)

events = events.filter(pl.col("timestamp_dt") < CUTOFF_DATE)
item_props = item_props.filter(pl.col("timestamp_dt") < CUTOFF_DATE)

print("events shape after cutoff:", events.shape)
print("item_props shape after cutoff:", item_props.shape)

events shape after cutoff: (2712523, 6)
item_props shape after cutoff: (20275902, 5)


## 3.3. Извлечение available

In [39]:
available_df = (
    item_props.filter(pl.col("property") == "available")
    .with_columns(pl.col("value").cast(pl.Float64, strict=False).alias("value_num"))
    .drop_nulls("value_num")
    .sort(["itemid", "timestamp_dt"])
    .group_by("itemid")
    .tail(1)
    .select([pl.col("itemid"), pl.col("value_num").cast(pl.Int8).alias("available")])
)

## 3.4. Извлечение categoryid

In [40]:
category_df = (
    item_props.filter(pl.col("property") == "categoryid")
    .with_columns(pl.col("value").cast(pl.Float64, strict=False))
    .drop_nulls("value")
    .sort("timestamp_dt")
    .group_by("itemid")
    .agg(pl.col("value").last())
    .with_columns(pl.col("value").cast(pl.Int32).alias("categoryid"))
    .select(["itemid", "categoryid"])
)

category_df.head()

itemid,categoryid
i32,i32
242902,1070
84820,1070
154551,342
456302,1522
195325,64


## 3.5. Агрегация базовых item-features

Сразу делаем простые признаки по свойствам товара.

In [41]:
item_prop_counts = (
    item_props.group_by("itemid").len().rename({"len": "item_prop_count"})
)

item_unique_props = item_props.group_by("itemid").agg(
    pl.col("property").n_unique().alias("item_unique_prop_count")
)

item_features = (
    item_prop_counts.join(item_unique_props, on="itemid", how="full")
    .join(available_df, on="itemid", how="left")
    .join(category_df, on="itemid", how="left")
    .with_columns(pl.col("available").fill_null(0).cast(pl.Int8))
    .sort("itemid")
)

item_features.head()

itemid,item_prop_count,itemid_right,item_unique_prop_count,available,categoryid
i32,u32,i32,u32,i8,i32
0,45,0,28,0,209
1,86,1,35,0,1114
2,24,2,24,0,1305
3,46,3,29,0,1171
4,42,4,25,0,1038


## 3.6. Присоединение родительской категории

In [42]:
item_features = item_features.join(
    category_tree.rename({"parentid": "parent_categoryid"}),
    on="categoryid",
    how="left",
)

item_features.head()

itemid,item_prop_count,itemid_right,item_unique_prop_count,available,categoryid,parent_categoryid
i32,u32,i32,u32,i8,i32,f32
0,45,0,28,0,209,293.0
1,86,1,35,0,1114,113.0
2,24,2,24,0,1305,1214.0
3,46,3,29,0,1171,938.0
4,42,4,25,0,1038,1174.0


## 3.7. Фильтрация недоступных товаров

На этапе исследования модели можно сохранить оба варианта:

полный каталог
каталог для рекомендаций

In [43]:
available_count = item_features.filter(pl.col("available") == 1).height

total_items = item_features.select(pl.col("itemid").n_unique()).item()

print("Available items:", available_count)
print("Share of available items:", available_count / total_items)

Available items: 54010
Share of available items: 0.1295039239617027


## 3.8. Сохраняем промежуточные артефакты

In [44]:
# Определяем корень проекта
PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Директория для сохранения
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Пути к файлам
item_features_path = PROCESSED_DIR / "item_features_base.parquet"
events_path = PROCESSED_DIR / "events_preprocessed.parquet"

# Сохранение
item_features.write_parquet(item_features_path)
events.write_parquet(events_path)


# Безопасный вывод относительного пути
def rel_path(path, root):
    try:
        return path.relative_to(root)
    except ValueError:
        return path


print(f"Saved item_features -> {rel_path(item_features_path, PROJECT_ROOT)}")
print(f"Saved events -> {rel_path(events_path, PROJECT_ROOT)}")

Saved item_features -> data/processed/item_features_base.parquet
Saved events -> data/processed/events_preprocessed.parquet


## Выводы по предобработке данных

В рамках предобработки были выполнены следующие шаги:

* временные признаки (`timestamp`) приведены к формату `datetime`;
* удалён неполный хвост данных для исключения искажений в распределениях;
* из `item_properties` извлечены ключевые признаки товаров:

  * `available` — доступность товара;
  * `categoryid` — категория товара;
* построена агрегированная таблица `item_features`, включающая:

  * количество свойств товара;
  * число уникальных свойств;
  * категорию и родительскую категорию;
  * признак доступности товара.

В результате получена базовая таблица признаков товаров, которая будет использоваться на следующих этапах:

* генерация кандидатов;
* построение признаков;
* ранжирование.

---

## Использование Polars

Для этапа предобработки использовалась библиотека **Polars**, что обусловлено следующими причинами:

* высокая скорость обработки данных по сравнению с pandas;
* эффективное использование памяти;
* удобный декларативный API для операций:

  * фильтрации;
  * агрегации (`group_by`);
  * соединений (`join`);
  * преобразования типов.

Polars особенно эффективен на этапах:

* загрузки данных;
* агрегаций;
* feature engineering.

При этом для этапа моделирования (обучение LightGBM, ALS) возможно использование pandas, так как большинство ML-библиотек работают с ним напрямую.

---

## Итог

Предобработка выполнена с учётом:

* предотвращения утечек данных;
* подготовки данных к задаче ранжирования;
* оптимизации вычислений.

Полученные данные готовы к следующему этапу — **разделению по времени и формированию таргета**.


# 4. Time-based split

Для корректной оценки рекомендательной системы используется разбиение по времени.

Это позволяет:
- исключить утечку будущей информации;
- приблизить offline-оценку к реальному production-сценарию;
- обучать модель только на прошлых событиях.

## 4.1. Смотрим временной диапазон

In [45]:
events.select(
    [
        pl.col("timestamp_dt").min().alias("min_timestamp"),
        pl.col("timestamp_dt").max().alias("max_timestamp"),
    ]
)

min_timestamp,max_timestamp
datetime[μs],datetime[μs]
2015-05-03 03:00:04.384,2015-09-14 23:59:45.566


## 4.2. Задаём границы сплитов

Для первого рабочего варианта предлагаю так:

train_end = 2015-08-15
valid_end = 2015-09-01
test = всё после valid_end до CUTOFF_DATE

Это даёт:

длинный train
отдельное validation окно
честный test

In [46]:
TRAIN_END = pl.datetime(2015, 8, 15)
VALID_END = pl.datetime(2015, 9, 1)
TEST_END = pl.datetime(2015, 9, 15)

## 4.3. Делаем сплиты

In [47]:
train_events = events.filter(pl.col("timestamp_dt") < TRAIN_END)

valid_events = events.filter(
    (pl.col("timestamp_dt") >= TRAIN_END) & (pl.col("timestamp_dt") < VALID_END)
)

test_events = events.filter(
    (pl.col("timestamp_dt") >= VALID_END) & (pl.col("timestamp_dt") < TEST_END)
)

print("train_events shape:", train_events.shape)
print("valid_events shape:", valid_events.shape)
print("test_events shape:", test_events.shape)

train_events shape: (2151115, 6)
valid_events shape: (301276, 6)
test_events shape: (260132, 6)


## 4.4. Проверяем диапазоны по каждому сплиту

In [48]:
def show_time_range(df, name):
    summary = df.select(
        [
            pl.col("timestamp_dt").min().alias("min_timestamp"),
            pl.col("timestamp_dt").max().alias("max_timestamp"),
            pl.len().alias("n_rows"),
        ]
    )
    print(f"\n{name}")
    print(summary)


show_time_range(train_events, "train")
show_time_range(valid_events, "valid")
show_time_range(test_events, "test")


train
shape: (1, 3)
┌─────────────────────────┬─────────────────────────┬─────────┐
│ min_timestamp           ┆ max_timestamp           ┆ n_rows  │
│ ---                     ┆ ---                     ┆ ---     │
│ datetime[μs]            ┆ datetime[μs]            ┆ u32     │
╞═════════════════════════╪═════════════════════════╪═════════╡
│ 2015-05-03 03:00:04.384 ┆ 2015-08-14 23:59:56.673 ┆ 2151115 │
└─────────────────────────┴─────────────────────────┴─────────┘

valid
shape: (1, 3)
┌─────────────────────────┬─────────────────────────┬────────┐
│ min_timestamp           ┆ max_timestamp           ┆ n_rows │
│ ---                     ┆ ---                     ┆ ---    │
│ datetime[μs]            ┆ datetime[μs]            ┆ u32    │
╞═════════════════════════╪═════════════════════════╪════════╡
│ 2015-08-15 00:00:07.484 ┆ 2015-08-31 23:59:58.647 ┆ 301276 │
└─────────────────────────┴─────────────────────────┴────────┘

test
shape: (1, 3)
┌─────────────────────────┬──────────────────────

## 4.5. Проверяем распределение событий по сплитам

Это важно, чтобы не получить, например, test без transaction или с перекошенной структурой.

In [49]:
def event_distribution(df, name):
    print(f"\n{name}")
    print(df.group_by("event").len().sort("len", descending=True))


event_distribution(train_events, "train")
event_distribution(valid_events, "valid")
event_distribution(test_events, "test")


train
shape: (3, 2)
┌─────────────┬─────────┐
│ event       ┆ len     │
│ ---         ┆ ---     │
│ cat         ┆ u32     │
╞═════════════╪═════════╡
│ view        ┆ 2080131 │
│ addtocart   ┆ 53529   │
│ transaction ┆ 17455   │
└─────────────┴─────────┘

valid
shape: (3, 2)
┌─────────────┬────────┐
│ event       ┆ len    │
│ ---         ┆ ---    │
│ cat         ┆ u32    │
╞═════════════╪════════╡
│ view        ┆ 290521 │
│ addtocart   ┆ 8122   │
│ transaction ┆ 2633   │
└─────────────┴────────┘

test
shape: (3, 2)
┌─────────────┬────────┐
│ event       ┆ len    │
│ ---         ┆ ---    │
│ cat         ┆ u32    │
╞═════════════╪════════╡
│ view        ┆ 251655 │
│ addtocart   ┆ 6463   │
│ transaction ┆ 2014   │
└─────────────┴────────┘


## 4.6. Проверяем число пользователей и товаров в каждом сплите

In [50]:
def split_stats(df, name):
    stats = df.select(
        [
            pl.len().alias("n_events"),
            pl.col("visitorid").n_unique().alias("n_users"),
            pl.col("itemid").n_unique().alias("n_items"),
        ]
    )
    print(f"\n{name}")
    print(stats)


split_stats(train_events, "train")
split_stats(valid_events, "valid")
split_stats(test_events, "test")


train
shape: (1, 3)
┌──────────┬─────────┬─────────┐
│ n_events ┆ n_users ┆ n_items │
│ ---      ┆ ---     ┆ ---     │
│ u32      ┆ u32     ┆ u32     │
╞══════════╪═════════╪═════════╡
│ 2151115  ┆ 1095383 ┆ 210687  │
└──────────┴─────────┴─────────┘

valid
shape: (1, 3)
┌──────────┬─────────┬─────────┐
│ n_events ┆ n_users ┆ n_items │
│ ---      ┆ ---     ┆ ---     │
│ u32      ┆ u32     ┆ u32     │
╞══════════╪═════════╪═════════╡
│ 301276   ┆ 171295  ┆ 81446   │
└──────────┴─────────┴─────────┘

test
shape: (1, 3)
┌──────────┬─────────┬─────────┐
│ n_events ┆ n_users ┆ n_items │
│ ---      ┆ ---     ┆ ---     │
│ u32      ┆ u32     ┆ u32     │
╞══════════╪═════════╪═════════╡
│ 260132   ┆ 150026  ┆ 75390   │
└──────────┴─────────┴─────────┘


## 4.7. Сохраняем сплиты

In [51]:
train_events_path = PROCESSED_DIR / "train_events.parquet"
valid_events_path = PROCESSED_DIR / "valid_events.parquet"
test_events_path = PROCESSED_DIR / "test_events.parquet"

train_events.write_parquet(train_events_path)
valid_events.write_parquet(valid_events_path)
test_events.write_parquet(test_events_path)

print(f"Saved train_events -> {train_events_path.relative_to(PROJECT_ROOT)}")
print(f"Saved valid_events -> {valid_events_path.relative_to(PROJECT_ROOT)}")
print(f"Saved test_events -> {test_events_path.relative_to(PROJECT_ROOT)}")

Saved train_events -> data/processed/train_events.parquet
Saved valid_events -> data/processed/valid_events.parquet
Saved test_events -> data/processed/test_events.parquet


### Выводы по time-based split

Данные разделены по времени на train, validation и test.

Такое разбиение:
- исключает использование будущих событий при обучении;
- соответствует реальному сценарию работы рекомендательной системы;
- позволяет корректно оценивать качество модели на новых данных.

Дальнейшее формирование таргета и признаков будет выполняться отдельно для каждого временного окна.

# 5. Формирование таргета

Таргет формируется на уровне пар (user, item).

Для каждого пользователя и товара:
- 1 — если пользователь добавил товар в корзину в будущем окне;
- 0 — иначе.

Используется временное разделение:
- история (features)
- будущее окно (target)

Это позволяет избежать утечки данных и приближает модель к реальному сценарию.

## 5.1. Определяем target события
используем:

* addtocart = основной таргет

* transaction можно тоже включить

In [52]:
TARGET_EVENTS = ["addtocart", "transaction"]

## 5.2. Формируем target для validation
делаем для valid (потом аналогично test)

In [53]:
valid_target = (
    valid_events.filter(pl.col("event").is_in(TARGET_EVENTS))
    .select(["visitorid", "itemid"])
    .unique()
    .with_columns(pl.lit(1).alias("target"))
)

valid_target.head()

visitorid,itemid,target
i32,i32,i32
247235,114149,1
426015,457119,1
718708,240558,1
1268072,27724,1
1245863,234222,1


## 5.3. Формируем кандидатов (простая версия)

Пока делаем простой вариант:

* берём товары, которые пользователь видел в train

In [54]:
train_user_items = (
    train_events
    .select(["visitorid", "itemid"])
    .unique()
)

## 5.4. Собираем user-item dataset

In [55]:
valid_dataset = (
    train_user_items
    .join(valid_target, on=["visitorid", "itemid"], how="left")
    .with_columns(
        pl.col("target").fill_null(0).cast(pl.Int8)
    )
)

valid_dataset.head()

visitorid,itemid,target
i32,i32,i8
364881,70810,0
761010,462203,0
931374,156592,0
995005,374358,0
988548,330844,0


## 5.5. Баланс классов (очень важно)

In [56]:
valid_dataset.select(
    [
        pl.len().alias("n_samples"),
        pl.col("target").sum().alias("positives"),
        (pl.col("target").sum() / pl.len()).alias("positive_rate"),
    ]
)

n_samples,positives,positive_rate
u32,i64,f64
1670835,261,0.000156


## Выводы по таргету и user-item dataset

### 1. Экстремальный дисбаланс классов

* доля позитивов: **~0.015%**
* 261 позитив на ~1.67 млн пар

* задача **очень разреженная и сложная**

---

### 2. Implicit feedback

* отсутствие `addtocart` ≠ негатив
* большинство нулей — это:

  * либо не видел
  * либо не заинтересовался

* классический implicit scenario

---

### 3. Ограниченное пространство кандидатов

* кандидаты = товары из train (исторические взаимодействия)
* нет новых товаров

* модель пока **не умеет рекомендовать новое**

---

### 4. Сильный сигнал таргета

* `addtocart` / `transaction` = высокое намерение
* это правильный выбор target

* хорошая постановка задачи

---

### 5. Датасет готов для ранжирования

* есть `(user, item, target)`
* есть temporal split
* нет leakage

* можно обучать:

* LightGBM ranker
* logistic baseline
* ALS + rerank

---

## Главный вывод

* текущая постановка:

```text
extreme imbalance + implicit feedback + sparse matrix
```

Это означает:

* accuracy бесполезна 
* нужен ranking подход 
* критичен candidate generation

---

## Что это означает для следующих шагов

Обязательно:

* negative sampling
* расширение кандидатов (ALS / popular)
* ranking модель (не классификация)

---

##  Самое важное

* сейчас главный bottleneck — **не модель, а кандидаты**